In [1]:
%pip install -U "transformers>=4.41" "datasets>=2.20" "accelerate>=0.33" "evaluate>=0.4" sentencepiece sacrebleu tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, random, numpy as np, torch, inspect
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)
import evaluate
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True

# Project & run knobs (kept small/fast)
PROJECT_DIR = "./marian_bilingual_en_es_pt"
os.makedirs(PROJECT_DIR, exist_ok=True)

MAX_SAMPLES = 1200        # per split, per language direction (train subset)
EVAL_SAMPLES = 300        # validation slice for quick BLEU
MAX_SRC_LEN = 128
MAX_TGT_LEN = 128
BATCH = 8                 # lower to 4/2 if OOM
GRAD_ACCUM = 1
MAX_STEPS = 500           # short fine-tune; bump to 1500+ for quality

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Device={device} | bf16={use_bf16} | fp16={use_fp16}")


Device=cuda | bf16=True | fp16=False


c:\Users\jeeva\anaconda3\Lib\site-packages\torch\__init__.py:1628: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:50.)
  _C._set_float32_matmul_precision(precision)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

# EN→ES (official)
tok_es = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es", token=None, local_files_only=False)
mod_es = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-es", token=None, local_files_only=False).to(device)

# EN→PT: try official, then community mirror, then a compact NLLB fallback
try:
    tok_pt = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-pt", token=None, local_files_only=False)
    mod_pt = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-pt", token=None, local_files_only=False).to(device)
    print("✅ Loaded Helsinki-NLP/opus-mt-en-pt")
except Exception as e1:
    print("⚠️ en-pt official failed:", type(e1).__name__)
    try:
        tok_pt = AutoTokenizer.from_pretrained("gsarti/opus-mt-en-pt", token=None, local_files_only=False)
        mod_pt = AutoModelForSeq2SeqLM.from_pretrained("gsarti/opus-mt-en-pt", token=None, local_files_only=False).to(device)
        print("✅ Loaded gsarti/opus-mt-en-pt")
    except Exception as e2:
        print("⚠️ mirror failed:", type(e2).__name__, "— using NLLB distilled as a fallback.")
        # NLLB fallback (works for many langs; slightly larger than Marian)
        tok_pt = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", token=None, local_files_only=False)
        mod_pt = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", token=None, local_files_only=False).to(device)
        # helper to translate to Portuguese with NLLB
        NLLB_CODE = {"pt": "por_Latn"}
        def translate_pt(text, max_new_tokens=128, num_beams=5):
            enc = tok_pt([text], return_tensors="pt", padding=True, truncation=True).to(device)
            gen = mod_pt.generate(
                **enc,
                forced_bos_token_id=tok_pt.convert_tokens_to_ids(NLLB_CODE["pt"]),
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )
            return tok_pt.batch_decode(gen, skip_special_tokens=True)[0]
        # define ES with Marian
        def translate_es(text, max_new_tokens=128, num_beams=5):
            e = tok_es([text], return_tensors="pt", padding=True, truncation=True).to(device)
            with torch.no_grad():
                g = mod_es.generate(**e, num_beams=num_beams, max_new_tokens=max_new_tokens, no_repeat_ngram_size=3, early_stopping=True)
            return tok_es.batch_decode(g, skip_special_tokens=True)[0]

#  define PT translator here:
if "translate_pt" not in globals():
    def translate_pt(text, max_new_tokens=128, num_beams=5):
        e = tok_pt([text], return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            g = mod_pt.generate(**e, num_beams=num_beams, max_new_tokens=max_new_tokens, no_repeat_ngram_size=3, early_stopping=True)
        return tok_pt.batch_decode(g, skip_special_tokens=True)[0]

def translate_es(text, max_new_tokens=128, num_beams=5):
    e = tok_es([text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        g = mod_es.generate(**e, num_beams=num_beams, max_new_tokens=max_new_tokens, no_repeat_ngram_size=3, early_stopping=True)
    return tok_es.batch_decode(g, skip_special_tokens=True)[0]

print(translate_es("The study focuses on electronic devices and materials."))
print(translate_pt("The study focuses on electronic devices and materials."))


⚠️ en-pt official failed: OSError
⚠️ mirror failed: OSError — using NLLB distilled as a fallback.


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

c:\Users\jeeva\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jeeva\.cache\huggingface\hub\models--facebook--nllb-200-distilled-600M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP d

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

El estudio se centra en dispositivos y materiales electrónicos.
O estudo concentra-se em dispositivos e materiais eletrônicos.


In [7]:
GLOSSARY = {
    "polymerase chain reaction": {"es": "reacción en cadena de la polimerasa", "pt": "reação em cadeia da polimerase"},
    "confidence interval": {"es": "intervalo de confianza", "pt": "intervalo de confiança"},
}
def apply_glossary(text: str, lang: str) -> str:
    out = text
    for en_term, mapping in GLOSSARY.items():
        if lang in mapping:
            out = out.replace(en_term, mapping[lang])
    return out


In [8]:
def load_small(pair: str, split: str, max_samples: int):
    """
    pair: 'en-es' or 'en-pt'
    returns Dataset with columns: src_text, tgt_text
    """
    ds = load_dataset("opus100", pair, split=split)
    ds = ds.select(range(min(max_samples, len(ds))))
    src_lang, tgt_lang = pair.split("-")
    def norm(ex):
        tr = ex.get("translation", {})
        return {"src_text": tr.get(src_lang, ex.get(src_lang)), "tgt_text": tr.get(tgt_lang, ex.get(tgt_lang))}
    return ds.map(norm, remove_columns=ds.column_names)

train_es = load_small("en-es", "train", MAX_SAMPLES)
valid_es = load_small("en-es", "validation", EVAL_SAMPLES)
train_pt = load_small("en-pt", "train", MAX_SAMPLES)
valid_pt = load_small("en-pt", "validation", EVAL_SAMPLES)

print("Train sizes  EN→ES / EN→PT:", len(train_es), len(train_pt))
print("Valid sizes  EN→ES / EN→PT:", len(valid_es), len(valid_pt))


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Train sizes  EN→ES / EN→PT: 1200 1200
Valid sizes  EN→ES / EN→PT: 300 300


In [10]:
# --- Robust preprocessor: works for MarianMT and NLLB tokenizers ---
LANG_CODE = {"es": "spa_Latn", "pt": "por_Latn", "en": "eng_Latn"}  # for NLLB

def preprocess_any(batch, tok, tgt_short: str):
    """
    - MarianMT: plain tokenization (no lang codes needed)
    - NLLB: set tok.tgt_lang = '<xxx_Latn>' before label tokenization
    - Use `text_target=` API when available; fallback to `as_target_tokenizer()`
    """
    # 1) optional glossary
    targets = [apply_glossary(t, tgt_short) for t in batch["tgt_text"]]

    # 2) inputs (same for both)
    model_inputs = tok(
        batch["src_text"],
        truncation=True,
        max_length=MAX_SRC_LEN,
    )

    # 3) labels
    is_nllb = "NllbTokenizer" in tok.__class__.__name__
    if is_nllb:
        # tell NLLB which target language to use
        tok.tgt_lang = LANG_CODE[tgt_short]

    # Prefer modern API (works in transformers >= 4.26); fallback for older versions
    try:
        labels = tok(
            text_target=targets,
            truncation=True,
            max_length=MAX_TGT_LEN,
        )
    except TypeError:
        # Older transformers: use context manager
        from contextlib import contextmanager
        cm = tok.as_target_tokenizer() if hasattr(tok, "as_target_tokenizer") else contextmanager(lambda: (yield))()
        with cm:
            labels = tok(
                targets,
                truncation=True,
                max_length=MAX_TGT_LEN,
            )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# ---- (re)tokenize with the robust preprocessor ----
train_es_tok = train_es.map(lambda b: preprocess_any(b, tok_es, "es"),
                            batched=True, remove_columns=train_es.column_names)
valid_es_tok = valid_es.map(lambda b: preprocess_any(b, tok_es, "es"),
                            batched=True, remove_columns=valid_es.column_names)

train_pt_tok = train_pt.map(lambda b: preprocess_any(b, tok_pt, "pt"),
                            batched=True, remove_columns=train_pt.column_names)
valid_pt_tok = valid_pt.map(lambda b: preprocess_any(b, tok_pt, "pt"),
                            batched=True, remove_columns=valid_pt.column_names)

print("✅ Tokenization done (ES & PT).")


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

✅ Tokenization done (ES & PT).


In [11]:
bleu = evaluate.load("sacrebleu")

def compute_metrics_for(tok):
    def _inner(eval_pred):
        preds, labels = eval_pred
        # decode preds
        decoded_preds = tok.batch_decode(preds, skip_special_tokens=True)
        # replace -100 in labels then decode
        labels = np.where(labels != -100, labels, tok.pad_token_id)
        decoded_labels = tok.batch_decode(labels, skip_special_tokens=True)
        return {"bleu": bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"]}
    return _inner

def make_training_args(**wanted):
    # filter kwargs to support older transformers
    accepted = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
    filtered = {k: v for k, v in wanted.items() if k in accepted}
    dropped = sorted(set(wanted) - set(filtered))
    if dropped: print("Dropped unsupported keys:", dropped)
    return TrainingArguments(**filtered)


In [15]:
# If not already imported at top:
from transformers import DataCollatorForSeq2Seq

# Build the seq2seq data collator for Marian EN→ES
collator_es = DataCollatorForSeq2Seq(
    tokenizer=tok_es,
    model=mod_es,
    padding="longest",   # or "max_length" if you prefer fixed padding
)

trainer_es = Trainer(
    model=mod_es,
    args=args_es,
    train_dataset=train_es_tok,
    eval_dataset=None,          # no eval during training
    data_collator=collator_es,
    tokenizer=tok_es,
    # compute_metrics=compute_metrics_for(tok_es),  # not needed without eval
)

print("🚀 Training Marian EN→ES (tiny)…")
trainer_es.train()
print("✅ EN→ES done.")


C:\Users\jeeva\AppData\Local\Temp\ipykernel_12016\3346243358.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_es = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 Training Marian EN→ES (tiny)…


Step,Training Loss
50,1.069400
100,1.001900
150,1.057000
200,0.778100
250,0.658300
300,0.611500
350,0.512000
400,0.550700
450,0.519100
500,0.426700


✅ EN→ES done.


In [ ]:
# If needed:
from transformers import DataCollatorForSeq2Seq

# Build the seq2seq data collator for EN→PT (works for Marian or NLLB)
collator_pt = DataCollatorForSeq2Seq(
    tokenizer=tok_pt,
    model=mod_pt,
    padding="longest",
)



trainer_pt = Trainer(
    model=mod_pt,
    args=args_pt,
    train_dataset=train_pt_tok,
    eval_dataset=None,          # eval later in a separate cell
    data_collator=collator_pt,
    tokenizer=tok_pt,
    # compute_metrics=compute_metrics_for(tok_pt),  # not needed without eval
)

print("🚀 Training Marian/NLLB EN→PT (tiny)…")
trainer_pt.train()
print("✅ EN→PT done.")



C:\Users\jeeva\AppData\Local\Temp\ipykernel_12016\1649804747.py:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_pt = Trainer(


🚀 Training Marian/NLLB EN→PT (tiny)…


Step,Training Loss
50,0.947800
100,1.141700
150,1.251800


In [ ]:
def batch_translate(texts, tok, mod, bs=16):
    outs = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            gen = mod.generate(**enc, num_beams=5, max_new_tokens=128, no_repeat_ngram_size=3, early_stopping=True)
        outs.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return outs

src_es = [x["src_text"] for x in load_small("en-es", "validation", EVAL_SAMPLES)]
ref_es = [[x["tgt_text"] for x in load_small("en-es", "validation", EVAL_SAMPLES)]]
src_pt = [x["src_text"] for x in load_small("en-pt", "validation", EVAL_SAMPLES)]
ref_pt = [[x["tgt_text"] for x in load_small("en-pt", "validation", EVAL_SAMPLES)]]

pred_es = batch_translate(src_es, tok_es, mod_es)
pred_pt = batch_translate(src_pt, tok_pt, mod_pt)

bleu = evaluate.load("sacrebleu")
score_es = bleu.compute(predictions=pred_es, references=ref_es)["score"]
score_pt = bleu.compute(predictions=pred_pt, references=ref_pt)["score"]

print(f"BLEU EN→ES (n={len(src_es)}): {score_es:.2f}")
print(f"BLEU EN→PT (n={len(src_pt)}): {score_pt:.2f}")

plt.figure()
plt.bar(["EN→ES", "EN→PT"], [score_es, score_pt])
plt.ylabel("BLEU")
plt.title("MarianMT tiny fine-tune — validation BLEU")
plt.show()


In [ ]:
def plot_loss(trainer, title):
    loss_points = [(e["step"], e["loss"]) for e in getattr(trainer.state, "log_history", []) if "loss" in e]
    if not loss_points:
        print(f"{title}: no loss logs found (set logging_steps > 0).")
        return
    steps, losses = zip(*loss_points)
    plt.figure()
    plt.plot(steps, losses)
    plt.xlabel("Step"); plt.ylabel("Training loss"); plt.title(title); plt.show()

plot_loss(trainer_es, "EN→ES: Training loss")
plot_loss(trainer_pt, "EN→PT: Training loss")


In [ ]:
def translate_es(text, max_new_tokens=128, num_beams=5):
    enc = tok_es([text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = mod_es.generate(**enc, num_beams=num_beams, max_new_tokens=max_new_tokens, no_repeat_ngram_size=3, early_stopping=True)
    return tok_es.batch_decode(out, skip_special_tokens=True)[0]

def translate_pt(text, max_new_tokens=128, num_beams=5):
    enc = tok_pt([text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = mod_pt.generate(**enc, num_beams=num_beams, max_new_tokens=max_new_tokens, no_repeat_ngram_size=3, early_stopping=True)
    return tok_pt.batch_decode(out, skip_special_tokens=True)[0]

print(translate_es("Polymerase chain reaction is widely used in molecular biology."))
print(translate_pt("Polymerase chain reaction is widely used in molecular biology."))
